# Lab

Build a service-desk workflow: waiting tickets use a bounded FIFO queue, and completed tickets use a LIFO history. Each task continues that design. Copy your completed methods into the next starter; every cell contains independent setup. Predict output, then test boundary behavior and explain your results. Sign in before submission and use Run C# for practice.


```{index} 1. record completed tickets
```

## 1. Record Completed Tickets



Complete `Push` and `Pop`. Expect latest T-102, then restoration order T-102, T-101, T-100. Trace count/capacity for the three pushes. Explain why pop clears the old top slot. Test empty pop, duplicate IDs, and reuse after emptying.


In [ ]:
using System;
using System.Collections.Generic;
var completed = new ArrayStack<string>();
completed.Push("T-100"); completed.Push("T-101"); completed.Push("T-102");
Console.WriteLine($"latest={completed.Peek()}, count={completed.Count}");
while (completed.TryPop(out string ticket)) Console.WriteLine($"restore={ticket}");
Console.WriteLine($"empty={completed.Count == 0}");
public class ArrayStack<T>
{
    private T[] items = new T[2];
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public void Push(T value)
    {
        // Your code starts here
        /* TODO: grow if full; store at Count; increase Count once. */
        // Your code ends here
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Stack is empty.");
        return items[Count - 1];
    }
    public T Pop()
    {
        // Your code starts here
        throw new NotImplementedException();
        // Your code ends here
    }
    public bool TryPop(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Pop();
        return true;
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
var completed = new ArrayStack<string>();
completed.Push("T-100"); completed.Push("T-101"); completed.Push("T-102");
Console.WriteLine($"latest={completed.Peek()}, count={completed.Count}");
while (completed.TryPop(out string ticket)) Console.WriteLine($"restore={ticket}");
Console.WriteLine($"empty={completed.Count == 0}");
public class ArrayStack<T>
{
    private T[] items = new T[2];
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public void Push(T value)
    {
        if (Count == Capacity)
        {
            var larger = new T[Capacity * 2];
            Array.Copy(items, larger, Count);
            items = larger;
        }
        items[Count++] = value;
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Stack is empty.");
        return items[Count - 1];
    }
    public T Pop()
    {
        T value = Peek();
        items[--Count] = default!;
        return value;
    }
    public bool TryPop(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Pop();
        return true;
    }
}

latest=T-102, count=3
restore=T-102
restore=T-101
restore=T-100
empty=True


```{index} 2. bound the waiting line
```

## 2. Bound the Waiting Line



Complete `Enqueue` using `(front + Count) % Capacity`. Capacity three must reject the fourth arrival, leaving T-100 at the front and count three. Give a deliberate overflow policy for a real service desk. Test capacity one and invalid constructor capacity.


In [ ]:
using System;
using System.Collections.Generic;
var waiting = new RingQueue<string>(3);
waiting.Enqueue("T-100"); waiting.Enqueue("T-101"); waiting.Enqueue("T-102");
try { waiting.Enqueue("T-103"); }
catch (InvalidOperationException) { Console.WriteLine("full request rejected"); }
Console.WriteLine($"front={waiting.Peek()}, count={waiting.Count}");
public class RingQueue<T>
{
    private readonly T[] items;
    private int front;
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public RingQueue(int capacity)
    {
        if (capacity <= 0) throw new ArgumentOutOfRangeException(nameof(capacity));
        items = new T[capacity];
    }
    public void Enqueue(T value)
    {
        // Your code starts here
        throw new NotImplementedException();
        // Your code ends here
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Queue is empty.");
        return items[front];
    }
    public T Dequeue()
    {
        T value = Peek();
        items[front] = default!;
        front = (front + 1) % Capacity;
        Count--;
        return value;
    }
    public bool TryDequeue(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Dequeue();
        return true;
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
var waiting = new RingQueue<string>(3);
waiting.Enqueue("T-100"); waiting.Enqueue("T-101"); waiting.Enqueue("T-102");
try { waiting.Enqueue("T-103"); }
catch (InvalidOperationException) { Console.WriteLine("full request rejected"); }
Console.WriteLine($"front={waiting.Peek()}, count={waiting.Count}");
public class RingQueue<T>
{
    private readonly T[] items;
    private int front;
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public RingQueue(int capacity)
    {
        if (capacity <= 0) throw new ArgumentOutOfRangeException(nameof(capacity));
        items = new T[capacity];
    }
    public void Enqueue(T value)
    {
        if (Count == Capacity) throw new InvalidOperationException("Queue is full.");
        int back = (front + Count) % Capacity;
        items[back] = value;
        Count++;
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Queue is empty.");
        return items[front];
    }
    public T Dequeue()
    {
        T value = Peek();
        items[front] = default!;
        front = (front + 1) % Capacity;
        Count--;
        return value;
    }
    public bool TryDequeue(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Dequeue();
        return true;
    }
}

full request rejected
front=T-100, count=3


```{index} 3. serve across the array boundary
```

## 3. Serve Across the Array Boundary



Complete `Dequeue`. Expect T-100, T-101, T-102, T-103 in service order, then reuse with T-104. Draw physical slots, front, and count after each operation. Explain why `front == back` alone cannot distinguish full from empty. Test failed dequeue without mutation.


In [ ]:
using System;
using System.Collections.Generic;
var waiting = new RingQueue<string>(3);
waiting.Enqueue("T-100"); waiting.Enqueue("T-101"); waiting.Enqueue("T-102");
Console.WriteLine($"serve={waiting.Dequeue()}");
waiting.Enqueue("T-103");
while (waiting.TryDequeue(out string ticket)) Console.WriteLine($"serve={ticket}");
waiting.Enqueue("T-104"); Console.WriteLine($"reused={waiting.Dequeue()}");
public class RingQueue<T>
{
    private readonly T[] items;
    private int front;
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public RingQueue(int capacity)
    {
        if (capacity <= 0) throw new ArgumentOutOfRangeException(nameof(capacity));
        items = new T[capacity];
    }
    public void Enqueue(T value)
    {
        if (Count == Capacity) throw new InvalidOperationException("Queue is full.");
        int back = (front + Count) % Capacity;
        items[back] = value;
        Count++;
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Queue is empty.");
        return items[front];
    }
    public T Dequeue()
    {
        // Your code starts here
        throw new NotImplementedException();
        // Your code ends here
    }
    public bool TryDequeue(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Dequeue();
        return true;
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
var waiting = new RingQueue<string>(3);
waiting.Enqueue("T-100"); waiting.Enqueue("T-101"); waiting.Enqueue("T-102");
Console.WriteLine($"serve={waiting.Dequeue()}");
waiting.Enqueue("T-103");
while (waiting.TryDequeue(out string ticket)) Console.WriteLine($"serve={ticket}");
waiting.Enqueue("T-104"); Console.WriteLine($"reused={waiting.Dequeue()}");
public class RingQueue<T>
{
    private readonly T[] items;
    private int front;
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public RingQueue(int capacity)
    {
        if (capacity <= 0) throw new ArgumentOutOfRangeException(nameof(capacity));
        items = new T[capacity];
    }
    public void Enqueue(T value)
    {
        if (Count == Capacity) throw new InvalidOperationException("Queue is full.");
        int back = (front + Count) % Capacity;
        items[back] = value;
        Count++;
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Queue is empty.");
        return items[front];
    }
    public T Dequeue()
    {
        T value = Peek();
        items[front] = default!;
        front = (front + 1) % Capacity;
        Count--;
        return value;
    }
    public bool TryDequeue(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Dequeue();
        return true;
    }
}

serve=T-100
serve=T-101
serve=T-102
serve=T-103
reused=T-104


```{index} 4. check the waiting-line contract
```

## 4. Check the Waiting-Line Contract



Replace the supplied queue methods with your completed versions. Write tests using the helpers: empty removal, duplicates, full rejection, alternating arrivals/service across wraparound, draining, and reuse. Check count and next ticket after failures, then drain to check every remaining value. Print the success line only after all checks pass. Compare four cycles with `Queue<T>`; this is a behavior check, not a speed benchmark.


In [ ]:
using System;
using System.Collections.Generic;
var waiting = new RingQueue<string>(3);
var expected = new Queue<string>();
void Check()
{
    if (waiting.Count != expected.Count) throw new Exception("Count mismatch");
    if (expected.Count > 0 && waiting.Peek() != expected.Peek()) throw new Exception("Front mismatch");
}
void Add(string ticket) { waiting.Enqueue(ticket); expected.Enqueue(ticket); Check(); }
void Serve()
{
    if (waiting.Dequeue() != expected.Dequeue()) throw new Exception("FIFO mismatch");
    Check();
}
void MustReject(Action request)
{
    try { request(); }
    catch (InvalidOperationException) { return; }
    throw new Exception("Boundary request was accepted");
}
// Your code starts here

// Your code ends here
public class RingQueue<T>
{
    private readonly T[] items;
    private int front;
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public RingQueue(int capacity)
    {
        if (capacity <= 0) throw new ArgumentOutOfRangeException(nameof(capacity));
        items = new T[capacity];
    }
    public void Enqueue(T value)
    {
        if (Count == Capacity) throw new InvalidOperationException("Queue is full.");
        int back = (front + Count) % Capacity;
        items[back] = value;
        Count++;
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Queue is empty.");
        return items[front];
    }
    public T Dequeue()
    {
        T value = Peek();
        items[front] = default!;
        front = (front + 1) % Capacity;
        Count--;
        return value;
    }
    public bool TryDequeue(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Dequeue();
        return true;
    }
}

In [1]:
// Solution
using System;
using System.Collections.Generic;
var waiting = new RingQueue<string>(3);
var expected = new Queue<string>();
void Check()
{
    if (waiting.Count != expected.Count) throw new Exception("Count mismatch");
    if (expected.Count > 0 && waiting.Peek() != expected.Peek()) throw new Exception("Front mismatch");
}
void Add(string ticket) { waiting.Enqueue(ticket); expected.Enqueue(ticket); Check(); }
void Serve()
{
    if (waiting.Dequeue() != expected.Dequeue()) throw new Exception("FIFO mismatch");
    Check();
}
void MustReject(Action request)
{
    try { request(); }
    catch (InvalidOperationException) { return; }
    throw new Exception("Boundary request was accepted");
}
MustReject(() => waiting.Dequeue()); Check();
for (int cycle = 0; cycle < 4; cycle++)
{
    Add("A"); Add("A"); Add("B");
    MustReject(() => waiting.Enqueue("LOST")); Check();
    Serve(); Add("C"); Serve(); Serve(); Serve();
    MustReject(() => waiting.Peek()); Check();
}
Console.WriteLine("FIFO, boundary, and reuse checks passed.");
public class RingQueue<T>
{
    private readonly T[] items;
    private int front;
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public RingQueue(int capacity)
    {
        if (capacity <= 0) throw new ArgumentOutOfRangeException(nameof(capacity));
        items = new T[capacity];
    }
    public void Enqueue(T value)
    {
        if (Count == Capacity) throw new InvalidOperationException("Queue is full.");
        int back = (front + Count) % Capacity;
        items[back] = value;
        Count++;
    }
    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("Queue is empty.");
        return items[front];
    }
    public T Dequeue()
    {
        T value = Peek();
        items[front] = default!;
        front = (front + 1) % Capacity;
        Count--;
        return value;
    }
    public bool TryDequeue(out T value)
    {
        if (Count == 0) { value = default!; return false; }
        value = Dequeue();
        return true;
    }
}

FIFO, boundary, and reuse checks passed.


```{index} 5. restore a completion and explain the policy
```

## 5. Restore a Completion and Explain the Policy



Write `RestoreLatest` using the completed stack and a waiting deque. Expect restoration of T-101 at the front, leaving `T-101, T-102`; later restoration empties the completion history. Define why this workflow restores at the front instead of the back. Test empty history and repeated serve/restore. Discuss whether unlimited restorations could postpone other tickets, and give a policy that limits that risk.


In [ ]:
using System;
using System.Collections.Generic;
var waiting = new LinkedList<string>(new[] { "T-100", "T-101", "T-102" });
var completed = new Stack<string>();
string ServeNext()
{
    if (waiting.First is null) throw new InvalidOperationException("No waiting tickets.");
    string ticket = waiting.First.Value;
    waiting.RemoveFirst(); completed.Push(ticket); return ticket;
}
bool RestoreLatest()
{
        // Your code starts here
        throw new NotImplementedException();
        // Your code ends here
    }
Console.WriteLine($"served={ServeNext()}"); Console.WriteLine($"served={ServeNext()}");
Console.WriteLine($"restored={RestoreLatest()}");
Console.WriteLine($"waiting={string.Join(", ", waiting)}");
RestoreLatest(); Console.WriteLine($"empty history={RestoreLatest() == false}");

In [1]:
// Solution
using System;
using System.Collections.Generic;
var waiting = new LinkedList<string>(new[] { "T-100", "T-101", "T-102" });
var completed = new Stack<string>();
string ServeNext()
{
    if (waiting.First is null) throw new InvalidOperationException("No waiting tickets.");
    string ticket = waiting.First.Value;
    waiting.RemoveFirst(); completed.Push(ticket); return ticket;
}
bool RestoreLatest()
{
    if (!completed.TryPop(out string? ticket)) return false;
    waiting.AddFirst(ticket); return true;
}
Console.WriteLine($"served={ServeNext()}"); Console.WriteLine($"served={ServeNext()}");
Console.WriteLine($"restored={RestoreLatest()}");
Console.WriteLine($"waiting={string.Join(", ", waiting)}");
RestoreLatest(); Console.WriteLine($"empty history={RestoreLatest() == false}");

served=T-100
served=T-101
restored=True
waiting=T-101, T-102
empty history=True


```{index} submit
```

## Submit



Submit all five coding responses, count/capacity and ring traces, boundary-test results, and your overflow/restoration policy explanations through the assignment submit panel. For local VS Code work, run `dotnet run` and use the instructor's configured submission method.
